# Проверка на своих картинках

Модель — обычный классификатор: softmax по классам «реальная» и семействам генераторов (OpenAI, Google, Flux,
Qwen и так далее). Оценка картинки — вероятность, что она сгенерирована: 1 − p(реальная). Выше 0.5 значит
сгенерирована, семейство — самый вероятный из остальных классов.

Порога рядом с моделью нет: после обучения смещение класса «реальная» подобрано так, что выше 0.5 оказывается
около 1 % реальных картинок валидации.

На CPU одна картинка занимает около 0.2 секунды.

In [ ]:
import os, io, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

from pathlib import Path
import numpy as np
import yaml, torch
import matplotlib.pyplot as plt
from PIL import Image
from src.model import Detector

run = Path('runs/base')   # папка запуска: model.pt и config.yaml
size = yaml.safe_load(open(run / 'config.yaml'))['train']['size']   # 512
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = Detector.load(run / 'model.pt', device)
model.classes   # семейства

Картинка готовится как на тесте: не-JPEG один раз сжимается в JPEG с качеством 90 (на обучении так сжимались
все не-JPEG), потом картинка целиком сжимается до 512×512.

In [ ]:
def prepare(path):
    image = Image.open(path)
    fmt, image = image.format, image.convert('RGB')
    if fmt != 'JPEG':
        buf = io.BytesIO()
        image.save(buf, 'JPEG', quality=90)
        image = Image.open(buf).convert('RGB')
    image = image.resize((size, size), Image.BICUBIC)
    return torch.from_numpy(np.array(image, dtype=np.float32) / 255).permute(2, 0, 1)   # [3, 512, 512], от 0 до 1


@torch.no_grad()
def predict(path):
    x = prepare(path)[None].to(device)
    with torch.autocast('cuda', dtype=torch.bfloat16, enabled=device == 'cuda'):
        logits = model(x)[0].float()              # [реальная, семейства]
    score = 1 - logits.softmax(0)[0].item()       # вероятность, что картинка сгенерирована
    share = logits[1:].softmax(0)                 # вероятности семейств, если она сгенерирована
    return score, {name: share[j].item() for j, name in enumerate(model.classes)}

In [ ]:
path = 'my_images/example.jpg'   # своя картинка

score, result = predict(path)
family = max(result, key=result.get)
verdict = f'сгенерирована ({score:.2f}), семейство {family}' if score > 0.5 else f'реальная ({score:.2f})'
print(verdict)

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].imshow(Image.open(path).convert('RGB'))
ax[0].set_title(verdict)
ax[0].axis('off')
order = sorted(result, key=result.get)
ax[1].barh(order, [result[n] for n in order], color='tab:red' if score > 0.5 else 'tab:gray')
ax[1].set_xlim(0, 1)
ax[1].set_title('семейство генератора, если картинка сгенерирована')
plt.tight_layout()
plt.show()

In [ ]:
# все картинки из папки
folder = Path('my_images')   # папка, не файл
for p in sorted(folder.iterdir()):
    if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}:
        score, result = predict(p)
        family = max(result, key=result.get) if score > 0.5 else ''
        print(f"{'FAKE' if score > 0.5 else 'real'}  {score:.2f}  {family:12s}  {p.name}")